# 27 — Pollution Accumulation Index

April 2025 replacement.

In [ ]:

CLEAR_SKY_CSV = "outputs/clear_sky_snapshot/clear_sky_shortlist.csv"
WIND_CALM_CSV = "outputs/wind_calm_test/wind_calm_candidate_days.csv"
EVENT_SUMMARY_CSV = "outputs/event_correlation/event_summary_by_day.csv"
OUTPUT_DIR = "outputs/accumulation_index"


In [ ]:

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, time, math, os
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars, retries=4, timeout=45):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {"latitude": lat, "longitude": lon, "start_date": start_date, "end_date": end_date, "hourly": hourly_vars, "timezone": "UTC"}
    last_err = None
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
    raise RuntimeError(f"Open-Meteo request failed after {retries} attempts: {last_err}")

def weather_cache_path(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    safe = f"{lat}_{lon}_{start_date}_{end_date}_{hourly_vars}".replace(",", "_").replace("/", "_").replace(":", "_")
    return cache_dir / f"{safe}.json"

def fetch_weather_cached(cache_dir: Path, lat, lon, start_date, end_date, hourly_vars):
    p = weather_cache_path(cache_dir, lat, lon, start_date, end_date, hourly_vars)
    if p.exists():
        return json.loads(p.read_text(encoding="utf-8"))
    j = fetch_weather_with_retry(lat, lon, start_date, end_date, hourly_vars)
    p.write_text(json.dumps(j), encoding="utf-8")
    return j

print("UTC now:", datetime.now(timezone.utc).isoformat())
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:

clear_sky,c_meta=safe_read_csv(CLEAR_SKY_CSV); wind_calm,w_meta=safe_read_csv(WIND_CALM_CSV); event_summary,e_meta=safe_read_csv(EVENT_SUMMARY_CSV)
frames=[]
if not clear_sky.empty: frames.append(clear_sky.assign(source_block="clear_sky"))
if not wind_calm.empty: frames.append(wind_calm.assign(source_block="wind_calm"))
core=pd.concat(frames,ignore_index=True,sort=False) if frames else pd.DataFrame(columns=["date","site_id","site_name"])
if "date" in core.columns: core["date"]=pd.to_datetime(core["date"],errors="coerce").dt.date.astype("string")
if not core.empty:
    summary=core.groupby(["date","site_id","site_name"],dropna=False).agg(
        mean_wind_speed_ms=("mean_wind_speed_ms","mean") if "mean_wind_speed_ms" in core.columns else ("date","size"),
        mean_cloud_cover_pct=("mean_cloud_cover_pct","mean") if "mean_cloud_cover_pct" in core.columns else ("date","size"),
        scene_count=("scene_count","max") if "scene_count" in core.columns else ("date","size"),
        clarity_score=("clarity_score","max") if "clarity_score" in core.columns else ("date","size"),
    ).reset_index()
else:
    summary=pd.DataFrame(columns=["date","site_id","site_name","mean_wind_speed_ms","mean_cloud_cover_pct","scene_count","clarity_score"])
if not event_summary.empty and "date" in event_summary.columns:
    event_summary["date"]=pd.to_datetime(event_summary["date"],errors="coerce").dt.date.astype("string")
    event_day=event_summary.groupby("date",dropna=False).agg(article_count=("article_count","max") if "article_count" in event_summary.columns else ("date","size"),forensic_interest_score=("forensic_interest_score","max") if "forensic_interest_score" in event_summary.columns else ("date","size")).reset_index()
    summary=summary.merge(event_day,on="date",how="left")
else:
    summary["article_count"]=0; summary["forensic_interest_score"]=0
for col in ["article_count","forensic_interest_score","mean_wind_speed_ms","mean_cloud_cover_pct","scene_count","clarity_score"]:
    if col not in summary.columns: summary[col]=0 if col in ["article_count","forensic_interest_score"] else np.nan
    summary[col]=pd.to_numeric(summary[col],errors="coerce")
summary["article_count"]=summary["article_count"].fillna(0); summary["forensic_interest_score"]=summary["forensic_interest_score"].fillna(0)
summary["accumulation_index"]=((5-summary["mean_wind_speed_ms"].fillna(5)).clip(lower=0)+(100-summary["mean_cloud_cover_pct"].fillna(100)).clip(lower=0)/25.0+summary["scene_count"].fillna(0)*0.5+summary["article_count"].fillna(0)*0.5+summary["forensic_interest_score"].fillna(0)*0.1)
summary.to_csv(Path(OUTPUT_DIR)/"pollution_accumulation_index.csv",index=False); display(summary.head(30))


In [ ]:

fig,ax=plt.subplots(figsize=(10,5))
if not summary.empty and "site_id" in summary.columns:
    for site in summary["site_id"].dropna().unique():
        s=summary[summary["site_id"]==site]; ax.plot(pd.to_datetime(s["date"]),s["accumulation_index"],marker="o",label=site)
ax.set_title("Pollution accumulation index"); ax.set_xlabel("Date"); ax.set_ylabel("Index")
if not summary.empty: ax.legend()
fig.autofmt_xdate(); fig.tight_layout()
plot_path=Path(OUTPUT_DIR)/"pollution_accumulation_index_plot.png"; fig.savefig(plot_path,dpi=150); plt.show()
(Path(OUTPUT_DIR)/"pollution_accumulation_index_manifest.json").write_text(json.dumps({"inputs":[c_meta,w_meta,e_meta],"rows_summary":int(len(summary))},indent=2),encoding="utf-8")
print("Saved:",plot_path)
